In [4]:
%display latex

#DEfinir parámetros
M = Manifold(4, 'M', r'\mathcal{M}')
X.<t, x, y, z> = M.chart(r't x y z')

# Variables
var('G', domain='real')
var('Lambda', domain='real')
var('c', domain='real')  
var('t', domain='real')

# DEfinir - Factores de escala
a1 = M.scalar_field(function('a1')(t), name='a1')
a2 = M.scalar_field(function('a2')(t), name='a2')
a3 = M.scalar_field(function('a3')(t), name='a3')

# Variables contenido de materia (fluido perfecto)
rho = M.scalar_field(function('rho')(t), name='rho')
p = M.scalar_field(function('p')(t), name='p')

# Métrica de Bianchi I 
g = M.metric('g', signature=-2)  # signatura para la métrica

# Métrica de Bianchi I
g[0,0] = 1    
g[1,1] = -a1^2
g[2,2] = -a2^2
g[3,3] = -a3^2

print("Métrica de Bianchi I:")
g.display()

Métrica de Bianchi I:


g = dt⊗dt - a1(t)^2 dx⊗dx - a2(t)^2 dy⊗dy - a3(t)^2 dz⊗dz

In [5]:
# 4-velocidad comóvil
u = M.vector_field(name='u')
u[0] = 1

# 1-forma asociada (bajar índice)
u_form = u.down(g)

print("1-forma u_μ:")
u_form.display()

# Verificar normalización: u^μ u_μ = 1
print("Norma u·u:")
(u_form(u)).expr()

1-forma u_μ:
Norma u·u:


1

In [6]:
g[:]

[       1        0        0        0]
[       0 -a1(t)^2        0        0]
[       0        0 -a2(t)^2        0]
[       0        0        0 -a3(t)^2]

In [7]:
u = M.vector_field('u')
u[0] = 1
u.display()

u = ∂/∂t

In [8]:
g(u,u).expr()

1

In [9]:
# Tensor de proyección sobre la hipersuperficie ortogonal a u
h = g - u_form*u_form

print("Proyector h_{μν}:")
h.display()

Proyector h_{μν}:


-a1(t)^2 dx⊗dx - a2(t)^2 dy⊗dy - a3(t)^2 dz⊗dz

In [11]:
# Verificar h_{μν} u^ν = 0
print("Contracción con la c-velocidad (ortogonal)")
(h.contract(u)).display()

Contracción con la c-velocidad


0

In [12]:
# Conexión de Levi-Civita
nab = g.connection()

# Derivada covariante de u_μ
nabla_u = nab(u_form)

#print("\text∇_ν u_μ:")
nabla_u.display()

∇_ν u_μ:


-a1(t)*d(a1)/dt dx⊗dx - a2(t)*d(a2)/dt dy⊗dy - a3(t)*d(a3)/dt dz⊗dz

In [18]:
# Extraer la expresión simbólica de cada factor de escala
a1_expr = a1.expr()
a2_expr = a2.expr()
a3_expr = a3.expr()


H1 = diff(a1_expr, t)/a1_expr
H2 = diff(a2_expr, t)/a2_expr
H3 = diff(a3_expr, t)/a3_expr


theta = (H1 + H2 + H3)

print("θ = H1+H2+H3")
show(theta)

θ = H1+H2+H3


diff(a1(t), t)/a1(t) + diff(a2(t), t)/a2(t) + diff(a3(t), t)/a3(t)

In [19]:
# Tensor de cizalladura
sigma = M.tensor_field(0, 2, name='sigma', sym=(0,1))

# σ_{μν} = ∇_ν u_μ - (1/3) θ h_{μν}
for i in range(4):
    sigma[i,i] = (nabla_u[i,i] - (theta/3)*h[i,i]).expr()

print("σ_{μν}:")
sigma.display()

σ_{μν}:


sigma = -1/3*(2*a1(t)*a2(t)*a3(t)*d(a1)/dt - a1(t)^2*a3(t)*d(a2)/dt - a1(t)^2*a2(t)*d(a3)/dt)/(a2(t)*a3(t)) dx⊗dx + 1/3*(a2(t)^2*a3(t)*d(a1)/dt - 2*a1(t)*a2(t)*a3(t)*d(a2)/dt + a1(t)*a2(t)^2*d(a3)/dt)/(a1(t)*a3(t)) dy⊗dy + 1/3*(a2(t)*a3(t)^2*d(a1)/dt + a1(t)*a3(t)^2*d(a2)/dt - 2*a1(t)*a2(t)*a3(t)*d(a3)/dt)/(a1(t)*a2(t)) dz⊗dz

In [22]:
# Componentes de la métrica inversa
g_inv = g.inverse()

# σ² = (1/2) σ_{μν} σ^{μν}
# Extraemos .expr() de cada factor para quedarnos con expresiones simbólicas
sigma2 = sum(
    (1/2) * (g_inv[i,i]).expr() * (g_inv[i,i]).expr() * (sigma[i,i]).expr()^2
    for i in range(4)
)

print("σ² =")
show(sigma2.simplify_full())

σ² =


1/3*(a2(t)^2*a3(t)^2*diff(a1(t), t)^2 - a1(t)*a2(t)*a3(t)^2*diff(a1(t), t)*diff(a2(t), t) + a1(t)^2*a3(t)^2*diff(a2(t), t)^2 + a1(t)^2*a2(t)^2*diff(a3(t), t)^2 - (a1(t)*a2(t)^2*a3(t)*diff(a1(t), t) + a1(t)^2*a2(t)*a3(t)*diff(a2(t), t))*diff(a3(t), t))/(a1(t)^2*a2(t)^2*a3(t)^2)

In [24]:
# Forma esperada: σ² = (1/6)[(H1-H2)² + (H2-H3)² + (H3-H1)²]
sigma2_esperado = (1/6)*((H1-H2)^2 + (H2-H3)^2 + (H3-H1)^2)

print("σ² (Sage):")
show(sigma2.simplify_full())

print("σ² (esperado):")
show(sigma2_esperado.simplify_full())

print("Diferencia debe ser cero:")
show((sigma2 - sigma2_esperado).simplify_full())

σ² (Sage):


1/3*(a2(t)^2*a3(t)^2*diff(a1(t), t)^2 - a1(t)*a2(t)*a3(t)^2*diff(a1(t), t)*diff(a2(t), t) + a1(t)^2*a3(t)^2*diff(a2(t), t)^2 + a1(t)^2*a2(t)^2*diff(a3(t), t)^2 - (a1(t)*a2(t)^2*a3(t)*diff(a1(t), t) + a1(t)^2*a2(t)*a3(t)*diff(a2(t), t))*diff(a3(t), t))/(a1(t)^2*a2(t)^2*a3(t)^2)

σ² (esperado):


1/3*(a2(t)^2*a3(t)^2*diff(a1(t), t)^2 - a1(t)*a2(t)*a3(t)^2*diff(a1(t), t)*diff(a2(t), t) + a1(t)^2*a3(t)^2*diff(a2(t), t)^2 + a1(t)^2*a2(t)^2*diff(a3(t), t)^2 - (a1(t)*a2(t)^2*a3(t)*diff(a1(t), t) + a1(t)^2*a2(t)*a3(t)*diff(a2(t), t))*diff(a3(t), t))/(a1(t)^2*a2(t)^2*a3(t)^2)

Diferencia debe ser cero:


0

In [25]:
# Verificar: H1*H2 + H2*H3 + H3*H1 = θ²/3 - σ²
lhs = H1*H2 + H2*H3 + H3*H1
rhs = theta^2/3 - sigma2_esperado  # usamos la forma simplificada

print("H1H2+H2H3+H3H1 =")
show(lhs.expand())

print("θ²/3 - σ² =")
show(rhs.simplify_full())

print("Diferencia (debe ser 0):")
show((lhs - rhs).simplify_full())

H1H2+H2H3+H3H1 =


diff(a1(t), t)*diff(a2(t), t)/(a1(t)*a2(t)) + diff(a1(t), t)*diff(a3(t), t)/(a1(t)*a3(t)) + diff(a2(t), t)*diff(a3(t), t)/(a2(t)*a3(t))

θ²/3 - σ² =


(a3(t)*diff(a1(t), t)*diff(a2(t), t) + (a2(t)*diff(a1(t), t) + a1(t)*diff(a2(t), t))*diff(a3(t), t))/(a1(t)*a2(t)*a3(t))

Diferencia (debe ser 0):


0

In [29]:
# Calcular ω 
omega = M.tensor_field(0, 2, name='omega', antisym=(0,1))

for i in range(4):
    for j in range(4):
        omega[i,j] = (nabla_u[i,j].expr() - nabla_u[j,i].expr())/2

print("ω_{μν}=0")
omega.display()

ω_{μν}=0


omega = 0

In [32]:
# Construir a_μ componente a componente
acc = M.vector_field(name='a')   # o 1-forma según lo que necesites

# a_μ = u^ν ∇_ν u_μ
# Manualmente, componente por componente
for mu in range(4):
    acc[mu] = sum(nabla_u[mu, nu].expr() * u[nu].expr() for nu in range(4))

print("a_μ = 0")
acc.display()

a_μ = 0


a = 0